# 03 Run Full (Part 2 / RegGS)

Goal:

1. generate the **formal Part 2 config**;
2. keep the full exported sequence as input;
3. use RegGS internal `sample_rate` / `n_views` for sparse-view protocol;
4. provide a clean notebook entry for running infer / refine / metric.

This notebook is the formal run entry after the smoke test passes.

Supports multiple datasets: Re10k-1, DL3DV-2 (set `DATASET_NAME` below).

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from pathlib import Path
import json
import subprocess
import shutil
import copy
import yaml

# ============== DATASET SELECTION ==============
# Options: 'Re10k-1', 'DL3DV-2'
DATASET_NAME = 'Re10k-1'
#DATASET_NAME = 'DL3DV-2'
# ===============================================

CV_ROOT = Path('~/CV_Project').expanduser()
REGGS_ROOT = CV_ROOT / 'third_party' / 'RegGS'
REGGS_PYTHON = Path('/home/bzhang512/miniconda3/envs/reggs/bin/python')
PART2_ROOT = CV_ROOT / 'part2'
CONFIG_ROOT = PART2_ROOT / 'configs'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'
RESULTS_ROOT = CV_ROOT / 'results' / 'part2'

# Scene path configuration based on dataset
SCENE_CONFIGS = {
    'Re10k-1': {
        'scene_name': 'reggs_re10k1_fullseq_256',
    },
    'DL3DV-2': {
        'scene_name': 'reggs_dl3dv2_fullseq_256',
    },
}

if DATASET_NAME not in SCENE_CONFIGS:
    raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME}. Options: {list(SCENE_CONFIGS.keys())}")

SCENE_ROOT = CV_ROOT / 'dataset' / DATASET_NAME / 'part2' / SCENE_CONFIGS[DATASET_NAME]['scene_name']
SCENE_PARENT = SCENE_ROOT.parent
SCENE_NAME = SCENE_ROOT.name

for d in [CONFIG_ROOT, OUTPUT_ROOT, RESULTS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print('DATASET_NAME =', DATASET_NAME)
print('REGGS_ROOT =', REGGS_ROOT)
print('REGGS_PYTHON =', REGGS_PYTHON)
print('SCENE_ROOT =', SCENE_ROOT)
print('SCENE_NAME =', SCENE_NAME)

DATASET_NAME = Re10k-1
REGGS_ROOT = /home/bzhang512/CV_Project/third_party/RegGS
REGGS_PYTHON = /home/bzhang512/miniconda3/envs/reggs/bin/python
SCENE_ROOT = /home/bzhang512/CV_Project/dataset/Re10k-1/part2/reggs_re10k1_fullseq_256
SCENE_NAME = reggs_re10k1_fullseq_256


## 1. Run configuration


In [4]:
# Formal Part 2 sparse protocol
FORMAL_SAMPLE_RATE = 50
FORMAL_N_VIEWS = 9
FORMAL_NEW_SUBMAP_EVERY = 2

# Aligner stability knobs (recommended start values for DL3DV; comments show RegGS defaults)
ALIGNER_ITERATIONS = 150         # default: 200
ALIGNER_CAM_ROT_LR = 1.0e-4      # default: 2.0e-4
ALIGNER_CAM_TRANS_LR = 1.0e-3    # default: 2.0e-3
ALIGNER_FILTER_ALPHA = True      # default: False
ALIGNER_ALPHA_THRE = 0.95        # default: 0.98
ALIGNER_MASK_INVALID_DEPTH = True  # default: False
ALIGNER_SCALE_MIN = 0.05         # default: 0.05
ALIGNER_SCALE_MAX = 20.0         # default: 20.0
ALIGNER_DEPTH_VALID_MIN = 0.05   # default: 0.05
ALIGNER_SCALE_INIT_FALLBACK = 1.0  # default: 1.0
ALIGNER_MW2_WARMUP_ITERS = 20    # default: 20

# Human-readable profile label for path/result separation when changing aligner knobs
RUN_PROFILE_TAG = 'comparison_check'

# Checkpoint selection policy: manual override > dataset mapping > unmapped fallback
MANUAL_NOPO_CHECKPOINT = 're10k'  # e.g. 're10k' / 'dl3dv' / 'acid' / None
DEFAULT_NOPO_CHECKPOINT_BY_DATASET = {
    'Re10k-1': 're10k',
    'DL3DV-2': 'dl3dv',
}
UNMAPPED_DATASET_CKPT_CANDIDATES = ['dl3dv', 're10k', 'acid']
VALID_NOPO_CHECKPOINTS = {'re10k', 'dl3dv', 'acid'}
NOPO_CHECKPOINT_TO_FILE = {
    're10k': 're10k.ckpt',
    'dl3dv': 'mixRe10kDl3dv.ckpt',
    'acid': 'acid.ckpt',
}

if MANUAL_NOPO_CHECKPOINT is not None:
    if MANUAL_NOPO_CHECKPOINT not in VALID_NOPO_CHECKPOINTS:
        raise ValueError(
            f'Invalid MANUAL_NOPO_CHECKPOINT={MANUAL_NOPO_CHECKPOINT}; '
            f'valid options are {sorted(VALID_NOPO_CHECKPOINTS)}'
        )
    FORMAL_NOPO_CHECKPOINT = MANUAL_NOPO_CHECKPOINT
    nopo_source = 'manual'
elif DATASET_NAME in DEFAULT_NOPO_CHECKPOINT_BY_DATASET:
    FORMAL_NOPO_CHECKPOINT = DEFAULT_NOPO_CHECKPOINT_BY_DATASET[DATASET_NAME]
    nopo_source = 'dataset_mapping'
else:
    existing_candidates = [
        ck for ck in UNMAPPED_DATASET_CKPT_CANDIDATES
        if (REGGS_ROOT / 'pretrained_weights' / NOPO_CHECKPOINT_TO_FILE[ck]).exists()
    ]
    if existing_candidates:
        FORMAL_NOPO_CHECKPOINT = existing_candidates[0]
        nopo_source = 'unmapped_dataset_existing_candidate_fallback'
    else:
        FORMAL_NOPO_CHECKPOINT = UNMAPPED_DATASET_CKPT_CANDIDATES[0]
        nopo_source = 'unmapped_dataset_candidate_fallback'
        print(
            f'[WARN] DATASET_NAME={DATASET_NAME} has no explicit mapping and no candidate ckpt file exists; '
            f'fallback to {FORMAL_NOPO_CHECKPOINT}.'
        )

NOPO_WEIGHT_PATH = REGGS_ROOT / 'pretrained_weights' / NOPO_CHECKPOINT_TO_FILE[FORMAL_NOPO_CHECKPOINT]
if not NOPO_WEIGHT_PATH.exists():
    print(f'[WARN] selected ckpt file not found: {NOPO_WEIGHT_PATH}')
else:
    print(f'[OK] selected ckpt file found: {NOPO_WEIGHT_PATH}')

# Generate dataset-specific run tag with key protocol knobs.
dataset_tag = DATASET_NAME.lower().replace('-', '')
profile_tag = RUN_PROFILE_TAG.strip() if RUN_PROFILE_TAG.strip() else 'default'
RUN_TAG = (
    f'reggs_{dataset_tag}_{FORMAL_NOPO_CHECKPOINT}-ckpt_'
    f'sr{FORMAL_SAMPLE_RATE}_nv{FORMAL_N_VIEWS}_sm{FORMAL_NEW_SUBMAP_EVERY}_{profile_tag}'
)
RUN_OUTPUT = OUTPUT_ROOT / DATASET_NAME.lower().replace('-', '_') / RUN_TAG
CONFIG_PATH = CONFIG_ROOT / f'{RUN_TAG}.yaml'

print('FORMAL_NOPO_CHECKPOINT =', FORMAL_NOPO_CHECKPOINT)
print('nopo selection source =', nopo_source)
print('RUN_PROFILE_TAG =', profile_tag)
print('RUN_TAG =', RUN_TAG)
print('RUN_OUTPUT =', RUN_OUTPUT)
print('CONFIG_PATH =', CONFIG_PATH)
print('ALIGNER knobs =', {
    'iterations': ALIGNER_ITERATIONS,
    'cam_rot_lr': ALIGNER_CAM_ROT_LR,
    'cam_trans_lr': ALIGNER_CAM_TRANS_LR,
    'filter_alpha': ALIGNER_FILTER_ALPHA,
    'alpha_thre': ALIGNER_ALPHA_THRE,
    'mask_invalid_depth': ALIGNER_MASK_INVALID_DEPTH,
    'scale_min': ALIGNER_SCALE_MIN,
    'scale_max': ALIGNER_SCALE_MAX,
    'depth_valid_min': ALIGNER_DEPTH_VALID_MIN,
    'scale_init_fallback': ALIGNER_SCALE_INIT_FALLBACK,
    'mw2_warmup_iters': ALIGNER_MW2_WARMUP_ITERS,
})


[OK] selected ckpt file found: /home/bzhang512/CV_Project/third_party/RegGS/pretrained_weights/re10k.ckpt
FORMAL_NOPO_CHECKPOINT = re10k
nopo selection source = manual
RUN_PROFILE_TAG = comparison_check
RUN_TAG = reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check
RUN_OUTPUT = /home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check
CONFIG_PATH = /home/bzhang512/CV_Project/part2/configs/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check.yaml
ALIGNER knobs = {'iterations': 150, 'cam_rot_lr': 0.0001, 'cam_trans_lr': 0.001, 'filter_alpha': True, 'alpha_thre': 0.95, 'mask_invalid_depth': True, 'scale_min': 0.05, 'scale_max': 20.0, 'depth_valid_min': 0.05, 'scale_init_fallback': 1.0, 'mw2_warmup_iters': 20}


## 2. Basic scene checks


In [5]:
assert SCENE_ROOT.exists(), f'Missing scene root: {SCENE_ROOT}'
assert (SCENE_ROOT / 'images').exists(), 'Missing images dir'
assert (SCENE_ROOT / 'intrinsics.json').exists(), 'Missing intrinsics.json'
assert (SCENE_ROOT / 'cameras.json').exists(), 'Missing cameras.json'

cameras = json.loads((SCENE_ROOT / 'cameras.json').read_text(encoding='utf-8'))
intrinsics = json.loads((SCENE_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
print('num frames =', len(cameras))
print('intrinsics =', intrinsics)


num frames = 279
intrinsics = {'fx': 0.8572737166601563, 'fy': 0.857483078, 'cx': 0.5, 'cy': 0.5}


## 3. Build formal config


In [6]:
base_cfg_path = REGGS_ROOT / 'config' / 're10k.yaml'
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))

cfg = copy.deepcopy(base_cfg)
cfg['dataset_name'] = 're10k'
cfg['n_views'] = int(FORMAL_N_VIEWS)
cfg['sample_rate'] = int(FORMAL_SAMPLE_RATE)
cfg['new_submap_every'] = int(FORMAL_NEW_SUBMAP_EVERY)
cfg['nopo_checkpoint'] = FORMAL_NOPO_CHECKPOINT
cfg['frame_limit'] = -1

cfg['data']['input_path'] = str(SCENE_PARENT)
cfg['data']['scene_name'] = SCENE_NAME
cfg['data']['output_path'] = str(RUN_OUTPUT)

cfg['cam']['H'] = 256
cfg['cam']['W'] = 256
cfg['cam']['depth_scale'] = 5000.0

cfg.setdefault('aligner', {})
cfg['aligner']['iterations'] = int(ALIGNER_ITERATIONS)
cfg['aligner']['cam_rot_lr'] = float(ALIGNER_CAM_ROT_LR)
cfg['aligner']['cam_trans_lr'] = float(ALIGNER_CAM_TRANS_LR)
cfg['aligner']['filter_alpha'] = bool(ALIGNER_FILTER_ALPHA)
cfg['aligner']['alpha_thre'] = float(ALIGNER_ALPHA_THRE)
cfg['aligner']['mask_invalid_depth'] = bool(ALIGNER_MASK_INVALID_DEPTH)
cfg['aligner']['scale_min'] = float(ALIGNER_SCALE_MIN)
cfg['aligner']['scale_max'] = float(ALIGNER_SCALE_MAX)
cfg['aligner']['depth_valid_min'] = float(ALIGNER_DEPTH_VALID_MIN)
cfg['aligner']['scale_init_fallback'] = float(ALIGNER_SCALE_INIT_FALLBACK)
cfg['aligner']['mw2_warmup_iters'] = int(ALIGNER_MW2_WARMUP_ITERS)

CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH.read_text(encoding='utf-8'))


dataset_name: re10k
checkpoint_path: null
frame_limit: -1
seed: 0
nopo_enable: true
n_views: 9
sample_rate: 50
nopo_checkpoint: re10k
new_submap_every: 2
aligner:
  iterations: 150
  cam_rot_lr: 0.0001
  cam_trans_lr: 0.001
  filter_alpha: true
  alpha_thre: 0.95
  soft_alpha: true
  mask_invalid_depth: true
  key_frame_pose: nopo
  scale_min: 0.05
  scale_max: 20.0
  depth_valid_min: 0.05
  scale_init_fallback: 1.0
  mw2_warmup_iters: 20
data:
  input_path: /home/bzhang512/CV_Project/dataset/Re10k-1/part2
  output_path: /home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check
  scene_name: reggs_re10k1_fullseq_256
cam:
  H: 256
  W: 256
  fx: 1
  fy: 1
  cx: 1
  cy: 1
  crop_edge: 0
  depth_scale: 5000.0



## 4. Optional cleanup


In [7]:
RESET_RUN_OUTPUT = True
if RESET_RUN_OUTPUT and RUN_OUTPUT.exists():
    shutil.rmtree(RUN_OUTPUT)
    print('removed old run output:', RUN_OUTPUT)
else:
    print('RESET_RUN_OUTPUT =', RESET_RUN_OUTPUT)


RESET_RUN_OUTPUT = True


## 5. Formal split preview


In [8]:
import numpy as np
frame_ids = np.arange(len(cameras))
test_ids = frame_ids[int(FORMAL_SAMPLE_RATE / 2)::FORMAL_SAMPLE_RATE]
remain_ids = np.array([i for i in frame_ids if i not in test_ids])
train_ids = remain_ids[np.linspace(0, remain_ids.shape[0] - 1, FORMAL_N_VIEWS).astype(int)]

print('train_ids =', train_ids.tolist())
print('test_ids =', test_ids.tolist())
print('num_train =', len(train_ids))
print('num_test =', len(test_ids))


train_ids = [0, 35, 69, 104, 139, 173, 208, 243, 278]
test_ids = [25, 75, 125, 175, 225, 275]
num_train = 9
num_test = 6


## 6. Run infer


In [9]:
RUN_INFER = True
infer_cmd = [str(REGGS_PYTHON), 'run_infer.py', str(CONFIG_PATH)]
print('infer_cmd =', ' '.join(infer_cmd))

if RUN_INFER:
    proc = subprocess.run(
        infer_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('infer returncode =', proc.returncode)
else:
    print('Set RUN_INFER=True to launch formal infer.')


infer_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_infer.py /home/bzhang512/CV_Project/part2/configs/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check.yaml
Using re10k.ckpt
/home/bzhang512/CV_Project/third_party/RegGS/src/utils/nopo_utils.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case whe

## 7. Run refine


In [10]:
RUN_REFINE = True
refine_cmd = [str(REGGS_PYTHON), 'run_refine.py', '--checkpoint_path', str(RUN_OUTPUT), '--config_path', str(CONFIG_PATH)]
print('refine_cmd =', ' '.join(refine_cmd))

if RUN_REFINE:
    proc = subprocess.run(
        refine_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('refine returncode =', proc.returncode)
else:
    print('Set RUN_REFINE=True after infer completes.')


refine_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_refine.py --checkpoint_path /home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check --config_path /home/bzhang512/CV_Project/part2/configs/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check.yaml
/home/bzhang512/CV_Project/third_party/RegGS/run_refine.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `t

## 8. Run metric


In [11]:
RUN_METRIC = True
METRIC_TEST_PROTOCOL = 'sampled_test'  # options: 'sampled_test' / 'all_non_train'
METRIC_OUTPUT_TAG = ''  # optional, e.g. 'all_non_train_v1'

if METRIC_TEST_PROTOCOL not in {'sampled_test', 'all_non_train'}:
    raise ValueError(
        f"Invalid METRIC_TEST_PROTOCOL={METRIC_TEST_PROTOCOL}; "
        "valid options are {'sampled_test', 'all_non_train'}"
    )

metric_cmd = [
    str(REGGS_PYTHON),
    'run_metric.py',
    '--checkpoint_path', str(RUN_OUTPUT),
    '--config_path', str(CONFIG_PATH),
    '--test_protocol', METRIC_TEST_PROTOCOL,
]
if METRIC_OUTPUT_TAG:
    metric_cmd += ['--test_output_tag', METRIC_OUTPUT_TAG]

print('metric_cmd =', ' '.join(metric_cmd))
print('METRIC_TEST_PROTOCOL =', METRIC_TEST_PROTOCOL)
print('METRIC_OUTPUT_TAG =', METRIC_OUTPUT_TAG)

if RUN_METRIC:
    proc = subprocess.run(
        metric_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('metric returncode =', proc.returncode)
else:
    print('Set RUN_METRIC=True after refine completes.')


metric_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_metric.py --checkpoint_path /home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check --config_path /home/bzhang512/CV_Project/part2/configs/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check.yaml --test_protocol sampled_test
METRIC_TEST_PROTOCOL = sampled_test
METRIC_OUTPUT_TAG = 


[Evaluator] Using gaussian file: /home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check/gaussians/global_refined_gs.ply
Training Frames: [  0  35  69 104 139 173 208 243 278]
Eval Frames (sampled_test): [ 25  75 125 175 225 275]
Evaluating train render...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/bzhang512/miniconda3/envs/reggs/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bzhang512/miniconda3/envs/reggs/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the m

## 9. Output check


In [12]:
print('run output exists =', RUN_OUTPUT.exists())
if RUN_OUTPUT.exists():
    for p in sorted(RUN_OUTPUT.iterdir())[:30]:
        print(' -', p.name)
else:
    print('No run output yet.')


run output exists = True
 - ate_aligned.json
 - config.yaml
 - estimated_c2w.ckpt
 - eval_test.json
 - eval_train.json
 - eval_traj.png
 - gaussians
 - gt
 - submaps
 - test
 - train
 - vis_align
 - vis_integrate
